# TB Drug-Resistance Data Preprocessing Pipeline

From raw mutation and phenotype data to the feature matrix **X** and the 18-drug resistance label matrix **Y**. All outputs are written to `outputs/`.

`phenotype.csv` is a given input, assembled from the merged drug tables and the online-database samples.

In [ ]:
import os
import pandas as pd
import numpy as np
import math
import re
from pathlib import Path

# ==================== Configuration ====================
# Root of the raw data. Override with the TB_DATA_ROOT environment variable
# if new_data_pipeline is not located next to this notebook's parent directory.
DATA_ROOT = Path(os.environ.get('TB_DATA_ROOT', '../new_data_pipeline'))
ALL_MUT   = DATA_ROOT / 'all_mut'
DRUG_DIR  = DATA_ROOT / 'drug'

OUTPUT_DIR = Path('outputs')            # all results written here
(OUTPUT_DIR / 'drug_raw_merged').mkdir(parents=True, exist_ok=True)

BASE_RATIO = 30 / 10000                  # rare-mutation threshold: 0.3% of samples

pd.set_option('display.width', 160)
print('DATA_ROOT :', DATA_ROOT, '| exists:', DATA_ROOT.exists())
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())

## Step 0 — Merge the paginated drug CSVs

Each drug folder holds paginated database exports (`data.csv`, `data (1).csv`, …), merged into one `<DRUG>_all.csv` per drug.

In [ ]:
for drug_dir in sorted(DRUG_DIR.iterdir()):
    if not drug_dir.is_dir() or drug_dir.name == 'extra':
        continue
    csvs = sorted(drug_dir.glob('data*.csv'))
    if not csvs:
        print(f'[{drug_dir.name}] no data*.csv, skipped')
        continue
    merged = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    merged.to_csv(OUTPUT_DIR / 'drug_raw_merged' / f'{drug_dir.name}_all.csv', index=False)
    print(f'[{drug_dir.name}] merged {len(csvs)} files -> {len(merged)} rows')

## Step 1 — Load, clean and merge the five mutation sources

- Sources: `literature`, `china_dataset`, `farhat_dataset`, `cryptic_dataset` (comma-separated multi-mutation strings, split with a whitelist regex), `extra_samples` (already formatted, counted under `cryptic_dataset`).
- Cleaning: lowercase / BOM-safe column names, drop rows with missing `uniqueid`, strip spaces and the `_clockwork` suffix, dedup on `(uniqueid, gene, mutation)`.
- Memory: only the required columns are read, and the phenotype / target-gene filters run per source before concatenation.

In [ ]:
FILE_CONFIG = {
    'literature':      ALL_MUT / 'literature_mutation_all.csv',
    'china_dataset':   ALL_MUT / 'china_mutation_all.csv',
    'farhat_dataset':  ALL_MUT / 'farhat_mutation_all.csv',
    'cryptic_dataset': ALL_MUT / 'cryptic_mutations_all.csv',
    'extra_samples':   ALL_MUT / 'extra_all_mutations.csv',
}
PHENOTYPE_FILE         = ALL_MUT / 'phenotype.csv'
INPUT_TARGET_GENE_FILE = ALL_MUT / 'gene_drug_tier1(original).csv'
CORE_COLS = ['gene', 'mutation', 'codes_protein', 'gene_position', 'indel_length']

valid_pheno_ids = (pd.read_csv(PHENOTYPE_FILE, usecols=['sra_accession'])['sra_accession']
                   .astype(str).str.strip().unique())
gene_drug_df = pd.read_csv(INPUT_TARGET_GENE_FILE)
target_genes_list = gene_drug_df['gene'].unique()
print(f'Phenotype sample list: {len(valid_pheno_ids)} | target genes: {len(target_genes_list)}')


def load_and_merge_files(file_config_dict):
    """Load, clean and merge the five mutation sources."""
    df_list, source_parts = [], []
    all_ids, kept_ids = set(), set()

    for custom_name, fpath in file_config_dict.items():
        if not fpath.exists():
            print(f'  -> [{custom_name}] file not found: {fpath}')
            continue

        # Read only the columns needed downstream (BOM-safe)
        needed = {'uniqueid', 'gene', 'mutation', 'mutations'} | set(CORE_COLS)
        df = pd.read_csv(fpath, usecols=lambda c: c.lstrip('\ufeff').lower() in needed)
        df.columns = [c.lstrip('\ufeff').lower() for c in df.columns]

        if 'uniqueid' not in df.columns:
            print(f'  -> [{custom_name}] ERROR: no uniqueid column, skipped')
            continue

        # Drop dirty rows with missing/blank uniqueid
        n0 = len(df)
        df['uniqueid'] = df['uniqueid'].replace(r'^\s*$', np.nan, regex=True)
        df.dropna(subset=['uniqueid'], inplace=True)
        if (dropped := n0 - len(df)) > 0:
            print(f'    [{custom_name}] dropped {dropped} rows with missing uniqueid')

        # ID cleaning: strip spaces and the _clockwork suffix
        df['uniqueid'] = df['uniqueid'].astype(str).str.strip()
        df['uniqueid'] = df['uniqueid'].str.replace('_clockwork', '', regex=False)

        # Source mapping: extra_samples counted under cryptic_dataset
        source_label = 'cryptic_dataset' if custom_name == 'extra_samples' else custom_name
        df['source_dataset'] = source_label

        # cryptic_dataset only: split comma-separated multi-mutation strings
        # (extra_samples is already formatted -> skipped)
        if custom_name == 'cryptic_dataset':
            target_col = 'mutations' if 'mutations' in df.columns else 'mutation'
            if target_col in df.columns:
                df[target_col] = df[target_col].astype(str).str.split(',')
                df = df.explode(target_col)
                df[target_col] = df[target_col].str.strip()
                # whitelist regex: PE_PGRS family / \d+kd_ag antigens / normal genes
                split = df[target_col].str.extract(r'^(PE_PGRS\d+|\d+kd_ag|[a-zA-Z0-9\-]+)_(.*)$')
                df['gene'], df['mutation'] = split[0], split[1]
                df.dropna(subset=['gene', 'mutation'], inplace=True)   # failed splits

        # Pad core columns so concat/apply never break
        for col in CORE_COLS:
            if col not in df.columns:
                df[col] = np.nan

        # Dedup within source (order-preserving)
        df.drop_duplicates(subset=['uniqueid', 'gene', 'mutation'], inplace=True)
        all_ids |= set(df['uniqueid'].unique())

        # Filter by phenotype sample list, snapshot the source mapping,
        # then filter to target genes
        df = df[df['uniqueid'].isin(valid_pheno_ids)]
        kept_ids |= set(df['uniqueid'].unique())
        source_parts.append(df[['uniqueid']].drop_duplicates().assign(source_dataset=source_label))
        df = df[df['gene'].isin(target_genes_list)]

        df_list.append(df[['uniqueid', 'source_dataset'] + CORE_COLS])
        print(f'  -> [{custom_name}] loaded as [{source_label}]: {len(df)} target-gene rows')

    if not df_list:
        return pd.DataFrame(), pd.DataFrame(), 0, 0

    merged_df = pd.concat(df_list, ignore_index=True)
    merged_df.drop_duplicates(subset=['uniqueid', 'gene', 'mutation'], inplace=True)
    source_map = (pd.concat(source_parts, ignore_index=True)
                  .drop_duplicates(subset=['uniqueid']))
    return merged_df, source_map, len(all_ids), len(kept_ids)


mutations_df, source_map, n_before, n_after = load_and_merge_files(FILE_CONFIG)

print(f'\nSamples in merged matrix before phenotype filter: {n_before}')
print(f'Samples removed (no phenotype record):             {n_before - n_after}')
print(f'Valid genotype samples kept:                      {n_after}')
print(f'Merged rows (phenotype + target-gene filtered):   {len(mutations_df)}')

## Step 2 — Variant classification

Each mutation is classified as `{gene}_coding_snp` / `{gene}_coding_indel` / `{gene}_coding_frameshift` / `{gene}_noncoding_snp` / `{gene}_noncoding_indel` or `synonymous` (e.g. `S450S`), using the explicit fields (`codes_protein`, `gene_position`, `indel_length`) plus regex fallbacks (`del|ins|fs` → indel; negative position → promoter; `S450L` / `S450*` → amino-acid SNP).

In [ ]:
filtered_df = mutations_df.copy()          # already phenotype- and gene-filtered
total_samples = filtered_df['uniqueid'].nunique()
print(f'Valid samples with target-gene mutations: {total_samples}')


def classify_variant(row):
    """Classify each mutation into a derived category."""
    gene = str(row.get('gene', ''))
    mutation = str(row.get('mutation', ''))

    # 1. Explicit fields
    is_coding = None
    if pd.notna(row.get('codes_protein')):
        val = str(row['codes_protein']).strip().lower()
        if val in ('true', '1', 'yes'):
            is_coding = True
        elif val in ('false', '0', 'no'):
            is_coding = False

    if is_coding is None and pd.notna(row.get('gene_position')):
        try:
            is_coding = (float(row['gene_position']) >= 0)
        except Exception:
            pass

    try:
        indel_len = float(row.get('indel_length', 0))
    except Exception:
        indel_len = 0
    is_indel_explicit = (indel_len != 0) and (not pd.isna(row.get('indel_length')))

    # 2. Regex fallback for missing annotations
    looks_like_indel     = bool(re.search(r'(del|ins|fs)', mutation.lower()))
    looks_like_promoter  = bool(re.search(r'(^|[a-zA-Z\.])-\d+', mutation)) or ('_-' in mutation)
    looks_like_amino_snp = bool(re.search(r'^[a-zA-Z]\d+[a-zA-Z*!]$', mutation))

    if is_coding is None:
        if looks_like_promoter:
            is_coding = False
        elif looks_like_amino_snp:
            is_coding = True
        else:
            is_coding = True   # default: most unannotated mutations are coding

    is_indel = is_indel_explicit or looks_like_indel

    # 3. Final category
    if is_coding:
        if is_indel:
            if is_indel_explicit:
                return f'{gene}_coding_indel' if abs(indel_len) % 3 == 0 else f'{gene}_coding_frameshift'
            return f'{gene}_coding_indel'
        # synonymous, e.g. S450S (same first/last letter)
        if len(mutation) > 1 and mutation[0] == mutation[-1] and mutation[0].isalpha():
            return 'synonymous'
        return f'{gene}_coding_snp'
    return f'{gene}_noncoding_indel' if is_indel else f'{gene}_noncoding_snp'


filtered_df['gene_mutation'] = filtered_df['gene'] + '_' + filtered_df['mutation']
filtered_df['derived_category'] = filtered_df.apply(classify_variant, axis=1)

print('Derived category distribution (top 15):')
print(filtered_df['derived_category'].value_counts().head(15))

## Step 3 — Build the feature matrix X

- **Common mutations** (count ≥ `max(2, ⌈0.3% × samples⌉)`): one 0/1 column each.
- **Rare mutations**: synonymous dropped, the rest aggregated by derived category; an aggregated column is kept only if its non-zero count also passes the threshold (second filter).

In [ ]:
# Dynamic rare-mutation threshold (0.3% of samples, minimum 2)
threshold = max(2, math.ceil(total_samples * BASE_RATIO))
print(f'Frequency threshold: >= {threshold} samples')

mutation_counts = filtered_df.groupby('gene_mutation')['uniqueid'].nunique()
common_muts_list = mutation_counts[mutation_counts >= threshold].index.tolist()
rare_muts_list   = mutation_counts[mutation_counts < threshold].index.tolist()
print(f'Common mutations (own column): {len(common_muts_list)}')
print(f'Rare mutations (to aggregate): {len(rare_muts_list)}')

# --- 3A. Common mutations: one 0/1 column each ---
df_common = filtered_df[filtered_df['gene_mutation'].isin(common_muts_list)]
X_common = pd.crosstab(df_common['uniqueid'], df_common['gene_mutation'])
X_common = (X_common > 0).astype(int)
print(f'Common-mutation matrix: {X_common.shape}')

# --- 3B. Rare mutations: drop synonymous, aggregate by derived category ---
df_rare = filtered_df[
    (filtered_df['gene_mutation'].isin(rare_muts_list)) &
    (filtered_df['derived_category'] != 'synonymous') &
    (filtered_df['derived_category'].notna())
]
if not df_rare.empty:
    X_rare = pd.crosstab(df_rare['uniqueid'], df_rare['derived_category'])
    X_rare = (X_rare > 0).astype(int)
    # second filter: aggregated column must also pass the threshold
    rare_cols_keep = [c for c in X_rare.columns if X_rare[c].sum() >= threshold]
    X_rare = X_rare[rare_cols_keep]
    print(f'Rare-mutation aggregated matrix: {X_rare.shape}')
else:
    X_rare = pd.DataFrame()

# --- 3C. Join ---
X_final = pd.concat([X_common, X_rare], axis=1).fillna(0).astype(int)
X_final.reset_index(inplace=True)
X_final.rename(columns={'index': 'uniqueid'}, inplace=True)
X_final['uniqueid'] = X_final['uniqueid'].astype(str).str.strip()
print(f'\nX feature matrix (before alignment): {X_final.shape}')

## Step 4 — Alignment and saving

- Gene-drug intermediate Y (sample × drug, 1 if the sample has any mutation in a gene linked to that drug) — intermediate product, **not** the final resistance labels.
- X and the intermediate Y are forced onto the same sample set and sorted by `uniqueid`, so row order matches 100%.
- Source relabel: `extra_samples` → `cryptic_dataset`; the sample-source mapping is saved separately.

In [ ]:
# --- 4A. Gene-drug intermediate Y ---
merged_labels_df = pd.merge(filtered_df, gene_drug_df[['gene', 'drug']], on='gene', how='left')
labels_matrix = pd.crosstab(merged_labels_df['uniqueid'], merged_labels_df['drug'])
labels_matrix = (labels_matrix > 0).astype(int)
labels_matrix.reset_index(inplace=True)
labels_matrix['uniqueid'] = labels_matrix['uniqueid'].astype(str).str.strip()

# --- 4B. Force X and Y onto the same sample set ---
target_ids = labels_matrix['uniqueid'].unique()
X_final = X_final.set_index('uniqueid').reindex(target_ids, fill_value=0).reset_index()
Y_final = labels_matrix.set_index('uniqueid').reindex(target_ids).reset_index()

X_final.sort_values('uniqueid', inplace=True)
Y_final.sort_values('uniqueid', inplace=True)

# --- 4C. Attach the source column, move meta columns to the front ---
X_final = pd.merge(X_final, source_map, on='uniqueid', how='left')
cols = X_final.columns.tolist()
if 'source_dataset' in cols:
    cols.remove('source_dataset')
    X_final = X_final[['uniqueid', 'source_dataset'] + [c for c in cols if c != 'uniqueid']]

# --- 4D. Source relabel: extra_samples -> cryptic_dataset ---
n_extra = (X_final['source_dataset'] == 'extra_samples').sum()
X_final['source_dataset'] = X_final['source_dataset'].replace('extra_samples', 'cryptic_dataset')
print(f'Source relabel: {n_extra} extra_samples -> cryptic_dataset')

# --- 4E. Save ---
assert len(X_final) == len(Y_final), 'X and Y row counts differ!'
X_final.to_csv(OUTPUT_DIR / 'X_feature_matrix_final.csv', index=False, encoding='utf-8-sig')
Y_final.to_csv(OUTPUT_DIR / 'Y_gene_drug_intermediate.csv', index=False, encoding='utf-8-sig')

# --- 4F. Sample-source mapping ---
X_final[['uniqueid', 'source_dataset']].to_csv(OUTPUT_DIR / 'sample_source_info.csv',
                                               index=False, encoding='utf-8-sig')

print(f'\nFinal samples: {len(X_final)}')
stats_df = pd.DataFrame({
    'N':      X_final['source_dataset'].value_counts(),
    'pct(%)': (X_final['source_dataset'].value_counts() / len(X_final) * 100).round(2),
})
print(stats_df.to_string())

## Step 5 — Build the 18-drug resistance label matrix Y

Labels come from `phenotype.csv`:

1. Keep only the 18 target drugs.
2. Binarize: **S (susceptible) → 0, R (resistant) → 1**; anything else → NaN.
3. Pivot long → wide (one row per sample, one column per drug).
4. Reindex to the X matrix sample order so X/Y rows align 100%.

In [ ]:
TARGET_DRUGS = ['AMK', 'BDQ', 'CAP', 'CFZ', 'CS', 'DLM', 'EMB', 'ETO', 'INH', 'KAN',
                'LFX', 'LZD', 'MFX', 'OFX', 'PZA', 'PAS', 'STM', 'RIF']

# Align to the X matrix sample order
ordered_ids = X_final['uniqueid'].astype(str).tolist()

df_pheno = pd.read_csv(PHENOTYPE_FILE, low_memory=False)
df_pheno.columns = df_pheno.columns.str.strip()
df_pheno['sra_accession'] = df_pheno['sra_accession'].astype(str).str.strip()
df_pheno['drug_abbr'] = df_pheno['drug_abbr'].astype(str).str.strip()

# 1) Keep the 18 target drugs
df_filtered = df_pheno[df_pheno['drug_abbr'].isin(TARGET_DRUGS)].copy()

# 2) Binarize: S -> 0, R -> 1, anything else -> NaN
df_filtered['status'] = df_filtered['binary_result'].astype(str).str.strip().map({'S': 0, 'R': 1})

# 3) Long -> wide
y_matrix = df_filtered.pivot_table(index='sra_accession', columns='drug_abbr', values='status')

# 4) Reindex to X order (missing records -> NaN)
y_matrix = y_matrix.reindex(ordered_ids)
y_matrix.index.name = 'uniqueid'
y_matrix = y_matrix.reset_index()

# 5) Save + per-drug statistics
y_matrix.to_csv(OUTPUT_DIR / 'Y_drug_labels_final.csv', index=False, encoding='utf-8-sig')

stats_data = []
for drug in TARGET_DRUGS:
    if drug in y_matrix.columns:
        counts = y_matrix[drug].value_counts(dropna=False)
        stats_data.append({'drug': drug,
                           'R (1)': int(counts.get(1.0, 0)),
                           'S (0)': int(counts.get(0.0, 0)),
                           'NaN':   int(y_matrix[drug].isna().sum())})
    else:
        stats_data.append({'drug': drug, 'R (1)': 0, 'S (0)': 0, 'NaN': len(y_matrix)})

print(f'Y matrix shape: {y_matrix.shape} (rows identical to X)')
print(pd.DataFrame(stats_data).to_string(index=False))